# LSTMs for Text Classification

**Dataset:** AG_NEWS (News topic classification: World, Sports, Business, Sci/Tech)

**Instructions:** Complete the simple `# TODO` sections marked with `None`. Run all cells top-to-bottom to train and evaluate your model.

In [1]:
# Run this cell to install dependencies and load the dataset
!pip install -q datasets

In [1]:
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Step 1: Vocabulary & Preprocessing
Neural networks need numbers, not raw text. We will build a vocabulary to map words to integers.

In [2]:
def tokenizer(text):
    return re.findall(r"[a-z0-9]+", text.lower())

ag_news = load_dataset('fancyzhx/ag_news')
train_data = ag_news['train']
test_data = ag_news['test']

def yield_tokens(data_iter):
    for example in data_iter:
        yield tokenizer(example['text'])

from collections import Counter

counter = Counter()
for tokens in yield_tokens(train_data):
    counter.update(tokens)

itos = ['<unk>', '<pad>'] + list(counter.keys())
stoi = {word: idx for idx, word in enumerate(itos)}
UNK_IDX = stoi['<unk>']
PAD_IDX = stoi['<pad>']

def numericalize(text):
    return [stoi.get(tok, UNK_IDX) for tok in tokenizer(text)]

print(f"Vocabulary size: {len(itos):,}")

def collate_batch(batch):
    label_list, text_list = [], []
    for example in batch:
        label_list.append(example['label'])  # ag_news labels are already 0-indexed (0-3)
        processed_text = torch.tensor(numericalize(example['text']), dtype=torch.int64)
        text_list.append(processed_text)

    # Pad text_list so all sentences in the batch are the same length.
    padded_texts = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor(label_list, dtype=torch.int64)

    return padded_texts, labels

# Using a small subset of data for fast training
train_list = list(train_data)[:5000]
test_list  = list(test_data)[:1000]

tr_ld = DataLoader(train_list, batch_size=32, shuffle=True, collate_fn=collate_batch)
te_ld = DataLoader(test_list, batch_size=32, shuffle=False, collate_fn=collate_batch)

C:\Program Files\Python312\Lib\importlib\__init__.py:90: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.5.1)
  return _bootstrap._gcd_import(name[level:], package, level)


Vocabulary size: 65,017


### Step 2: Build the LSTM Model
Construct a simple LSTM text classifier.

In [3]:
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # Define a PyTorch nn.LSTM layer (batch_first=True)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # Define a fully connected layer mapping hidden_dim to num_classes
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, text):
        embedded = self.embedding(text)

        # Pass the embedded text through the LSTM
        # The LSTM returns two things: output and (hidden_state, cell_state)
        output, (hidden, cell) = self.lstm(embedded)

        # We only care about the final hidden state of the last layer for classification
        final_hidden = hidden[-1]

        # Pass the final hidden state through the fully connected layer
        logits = self.fc(final_hidden)
        return logits

model = SimpleLSTM(vocab_size=len(itos), embed_dim=64, hidden_dim=128, num_classes=4).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 4,260,932


### Step 3: Train and Evaluate
Write the core steps of the PyTorch training loop.

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for texts, labels in tr_ld:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()

        # Perform a forward pass
        predictions = model(texts)

        # Compute the loss
        loss = criterion(predictions, labels)

        # Perform backpropagation
        loss.backward()

        # Clip gradients to prevent unstable updates
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update the weights
        optimizer.step()

        total_loss += loss.item()
        correct += (predictions.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # Evaluation Phase
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for texts, labels in te_ld:
            texts, labels = texts.to(device), labels.to(device)
            preds = model(texts)
            val_correct += (preds.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/10 | Train Acc: {train_acc:.1%} | Val Acc: {val_acc:.1%}")

Epoch 1/10 | Train Acc: 42.7% | Val Acc: 39.9%
Epoch 2/10 | Train Acc: 48.1% | Val Acc: 41.2%
Epoch 3/10 | Train Acc: 52.1% | Val Acc: 42.5%
Epoch 4/10 | Train Acc: 56.0% | Val Acc: 44.6%
Epoch 5/10 | Train Acc: 58.9% | Val Acc: 46.1%
Epoch 6/10 | Train Acc: 60.1% | Val Acc: 46.3%
Epoch 7/10 | Train Acc: 62.3% | Val Acc: 50.0%
Epoch 8/10 | Train Acc: 62.4% | Val Acc: 46.6%
Epoch 9/10 | Train Acc: 64.9% | Val Acc: 49.7%
Epoch 10/10 | Train Acc: 64.6% | Val Acc: 47.7%


### Step 4: Reflection
1. What does the `padding_idx` argument do in the `nn.Embedding` layer?
2. Why do we extract `hidden[-1]` instead of using the raw `output` from the LSTM for classification?

--

**1. Role of `padding_idx`**

Because sentences in a batch have different lengths, `collate_batch` pads shorter sequences with `PAD_IDX` so every sequence in a batch matches the length of the longest one. Passing `padding_idx=PAD_IDX` to `nn.Embedding` tells the layer to treat that specific index as a non informative filler: its embedding vector is initialized to all zeros and, critically, its gradient is not updated during backpropagation. Without this, the embedding for the pad token would drift during training just like any other word, letting the model learn spurious signal from a token that carries no real linguistic content — effectively injecting noise proportional to how much padding a given batch needed.

**2. Why `hidden[-1]` instead of the raw `output`**

`output` from `nn.LSTM` contains the hidden state produced at *every* timestep of the sequence — shape `(batch, seq_len, hidden_dim)`. For a single label classification task, we don't want a prediction per token; we want one summary vector per sentence. `hidden` (specifically `hidden[-1]`, the last layer's final hidden state) is exactly that: after processing the whole sequence step by step, it represents the LSTM's cumulative understanding of the entire input, folding earlier context into later timesteps. Using it as the input to the classifier is both simpler (fixed size vector, no pooling/flattening needed) and semantically correct, since it's the state explicitly designed to have "seen" the full sentence.
_____________

#### 3. Results across the three experiments

| Run | Change | Val Acc (final epoch) | Val Acc trend |
|---|---|---|---|
| 1 | `lr=0.005`, 3 epochs | 25.5% | Flat — stuck at random chance (25% baseline for 4 classes) |
| 2 | `lr=0.001`, 3 epochs | 32.4% | Rising, but too few epochs to see where it levels off |
| 3 | `lr=0.001` + gradient clipping, 10 epochs | 47.7% | Rises to a peak of **50.0% at epoch 7**, then declines |

Lowering the learning rate and adding gradient clipping fixed the original problem: the model went from not learning at all (Run 1) to actually acquiring real signal (Run 3), with train accuracy climbing from 42.7% to 64.6% and val accuracy roughly doubling versus Run 1. The inference sanity check also improved — the model no longer collapses to predicting a single class for every headline.

#### 4. Why the evaluation is still not good, overfitting has replaced underfitting

The remaining problem is visible in the **train/val gap**, which widens every epoch:

| Epoch | Train Acc | Val Acc | Gap |
|---|---|---|---|
| 1 | 42.7% | 39.9% | 2.8 pts |
| 5 | 58.9% | 46.1% | 12.8 pts |
| 7 | 62.3% | **50.0% (peak)** | 12.3 pts |
| 10 | 64.6% | 47.7% | 16.9 pts |

Val accuracy peaks at epoch 7 and then **drops** for the remaining three epochs, while train accuracy keeps climbing the whole time. That is the textbook signature of overfitting: the model is increasingly good at memorizing the 5,000 training examples it has seen, but that memorization stops transferring to unseen validation data past a certain point. This is a different failure mode from Runs 1 and 2 (which underfit ,the model wasn't learning enough), now the model is learning *too specifically* to the training set. This is also confirmed by the inference results: only 2 of 4 sample headlines were classified correctly, which is roughly consistent with a model performing around the 47-50% range rather than one that has generalized well.

#### 5. What we could do better

- **Early stopping now actually applicable.** Unlike the earlier flat/noisy runs, this run has a clear peak (epoch 7, 50.0% val acc) followed by decline. Restoring the model weights from epoch 7 instead of epoch 10 would recover the best generalization observed. This is the one change from the list I'd prioritize first, since it requires no retraining just checkpointing the best val accuracy model during training and reloading it at the end.
- **Regularization.** Add dropout (e.g., `nn.Dropout(0.3)` on the final hidden state before the FC layer) and/or weight decay in the optimizer (`torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)`) to directly discourage the model from over relying on training set specific patterns.
- **More training data.** 5,000 examples is small relative to a 65,017 word vocabulary being learned from scratch; the model has enough capacity (4.26M parameters) to memorize this subset well before it has seen enough variety to generalize. Increasing to 15,000–20,000 examples would give the model more signal to learn from before overfitting sets in.
- **Reduce model capacity.** Shrinking `hidden_dim` (e.g., 128 → 64) reduces how much the model can memorize outright, which is a natural complement to using more data.
- **Learning rate scheduling.** A scheduler that decays `lr` over epochs (e.g., `torch.optim.lr_scheduler.StepLR`) can let the model take larger, more exploratory steps early on and smaller, more refining steps later sometimes delaying the point where overfitting sets in.

### Step 5: Inference
Now that the model is trained, let's use it to classify a brand new headline that it has never seen before.

In [9]:
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

def predict(text, model):
    model.eval()

    # Convert the raw text into a tensor of token ids using numericalize().
    text_tensor = torch.tensor(numericalize(text), dtype=torch.int64).to(device)

    # The model expects a batch dimension. Add one with .unsqueeze(0).
    text_tensor = text_tensor.unsqueeze(0)

    with torch.no_grad():
        # Run a forward pass through the model to get the logits.
        logits = model(text_tensor)

        # Get the predicted class index from the logits (highest score).
        predicted_idx = logits.argmax(1).item()

    return class_names[predicted_idx]


sample_headlines = [
    "Manchester United wins dramatic final in extra time",
    "Central bank raises interest rates to combat inflation",
    "NASA's new telescope captures images of distant galaxy",
    "Peace talks resume between the two neighboring countries"
]

for headline in sample_headlines:
    prediction = predict(headline, model)
    print(f"'{headline}' -> {prediction}")

'Manchester United wins dramatic final in extra time' -> Business
'Central bank raises interest rates to combat inflation' -> Sci/Tech
'NASA's new telescope captures images of distant galaxy' -> Sci/Tech
'Peace talks resume between the two neighboring countries' -> World


## Retraining the model with above suggestion:

In [19]:
def tokenizer(text):
    return re.findall(r"[a-z0-9]+", text.lower())

ag_news = load_dataset('fancyzhx/ag_news')
train_data = ag_news['train']
test_data = ag_news['test']

def yield_tokens(data_iter):
    for example in data_iter:
        yield tokenizer(example['text'])

from collections import Counter

counter = Counter()
for tokens in yield_tokens(train_data):
    counter.update(tokens)

itos = ['<unk>', '<pad>'] + list(counter.keys())
stoi = {word: idx for idx, word in enumerate(itos)}
UNK_IDX = stoi['<unk>']
PAD_IDX = stoi['<pad>']

def numericalize(text):
    return [stoi.get(tok, UNK_IDX) for tok in tokenizer(text)]

print(f"Vocabulary size: {len(itos):,}")

def collate_batch(batch):
    label_list, text_list = [], []
    for example in batch:
        label_list.append(example['label'])  # ag_news labels are already 0-indexed (0-3)
        processed_text = torch.tensor(numericalize(example['text']), dtype=torch.int64)
        text_list.append(processed_text)

    # Pad text_list so all sentences in the batch are the same length.
    padded_texts = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor(label_list, dtype=torch.int64)

    return padded_texts, labels

# Using a small subset of data for fast training
train_list = list(train_data)[:100000]
test_list  = list(test_data)[:10000]

tr_ld = DataLoader(train_list, batch_size=32, shuffle=True, collate_fn=collate_batch)
te_ld = DataLoader(test_list, batch_size=32, shuffle=False, collate_fn=collate_batch)

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/datasets/fancyzhx/ag_news/resolve/eb185aade064a813bc0b7f42de02595523103ca4/ag_news.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since fancyzhx/ag_news couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\ACER\.cache\huggingface\datasets\fancyzhx___ag_news\default\0.0.0\eb185aade064a813bc0b7f42de02595523103ca4 (last modified on Fri Aug  7 10:20:01 2026).


Vocabulary size: 65,017


In [20]:
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # Define a PyTorch nn.LSTM layer (batch_first=True)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # Define a fully connected layer mapping hidden_dim to num_classes
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, text):
        embedded = self.embedding(text)

        # Pass the embedded text through the LSTM
        # The LSTM returns two things: output and (hidden_state, cell_state)
        output, (hidden, cell) = self.lstm(embedded)

        # We only care about the final hidden state of the last layer for classification
        final_hidden = hidden[-1]

        # Pass the final hidden state through the fully connected layer
        logits = self.fc(final_hidden)
        return logits

model = SimpleLSTM(vocab_size=len(itos), embed_dim=64, hidden_dim=128, num_classes=4).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 4,260,932


In [23]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for texts, labels in tr_ld:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()

        # Perform a forward pass
        predictions = model(texts)

        # Compute the loss
        loss = criterion(predictions, labels)

        # Perform backpropagation
        loss.backward()

        # Clip gradients to prevent unstable updates
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update the weights
        optimizer.step()

        total_loss += loss.item()
        correct += (predictions.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # Evaluation Phase
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for texts, labels in te_ld:
            texts, labels = texts.to(device), labels.to(device)
            preds = model(texts)
            val_correct += (preds.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/10 | Train Acc: {train_acc:.1%} | Val Acc: {val_acc:.1%}")

Epoch 1/10 | Train Acc: 99.2% | Val Acc: 89.9%
Epoch 2/10 | Train Acc: 99.4% | Val Acc: 90.0%
Epoch 3/10 | Train Acc: 99.5% | Val Acc: 89.7%


In [22]:
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

def predict(text, model):
    model.eval()

    # Convert the raw text into a tensor of token ids using numericalize().
    text_tensor = torch.tensor(numericalize(text), dtype=torch.int64).to(device)

    # The model expects a batch dimension. Add one with .unsqueeze(0).
    text_tensor = text_tensor.unsqueeze(0)

    with torch.no_grad():
        # Run a forward pass through the model to get the logits.
        logits = model(text_tensor)

        # Get the predicted class index from the logits (highest score).
        predicted_idx = logits.argmax(1).item()

    return class_names[predicted_idx]


sample_headlines = [
    "Manchester United wins dramatic final in extra time",
    "Central bank raises interest rates to combat inflation",
    "NASA's new telescope captures images of distant galaxy",
    "Peace talks resume between the two neighboring countries"
]

for headline in sample_headlines:
    prediction = predict(headline, model)
    print(f"'{headline}' -> {prediction}")

'Manchester United wins dramatic final in extra time' -> Sports
'Central bank raises interest rates to combat inflation' -> Business
'NASA's new telescope captures images of distant galaxy' -> Sci/Tech
'Peace talks resume between the two neighboring countries' -> World
